# LAB: RANDOM FOREST CLASSIFICATION

**Dataset:** `Iris.csv` — 150 samples, 3 species (*Iris-setosa*, *Iris-versicolor*, *Iris-virginica*)
**Features:** `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`
**Target:** `Species` (3 classes)

---
### Objectives
- Implement a Random Forest classifier from scratch: Bootstrap Sampling, Random Feature Subsampling, and Majority-Vote Aggregation.
- Apply Scikit-Learn's `RandomForestClassifier` with hyperparameter tuning, OOB evaluation, and cross-validation.
- Compare ensemble performance against a single Decision Tree baseline.

## PART 1: IMPLEMENTING RANDOM FOREST FROM SCRATCH

### 1.1. Theoretical Background

**Random Forest** is a bagging ensemble of $T$ Decision Trees.
Each tree $h_t$ is trained on a **bootstrap sample** $\mathcal{D}_t$
(drawn with replacement from the training set) and uses only a random
subset of $m$ features at each split:

$$m = \lfloor \sqrt{p} \rfloor \quad (\text{classification default, where } p = \text{total features})$$

**Key mechanisms:**

| Component | Description |
|-----------|-------------|
| **Bootstrap Sampling** | Each tree trains on $N$ samples drawn with replacement from $\mathcal{D}$ |
| **Random Feature Subset** | At each node, only $m$ of $p$ features are considered for splitting |
| **Majority Vote** | Final prediction = $\hat{y} = \arg\max_c \sum_t \mathbf{1}[h_t(x)=c]$ |

These two sources of randomness (sample + feature) decorrelate the trees,
reducing variance without increasing bias compared to a single deep tree.

### 1.2. Base Decision Tree (Provided)
The `Node` class and `DecisionTreeScratch` below serve as the base learner for the Random Forest.
These components are provided in full — your task in sections 1.3 and 1.4 is to implement the
**Random Forest-specific** mechanisms on top of them.

In [ ]:
import numpy as np

# ── Impurity helpers ──────────────────────────────────────────────────────
def _gini(y):
    if len(y) == 0: return 0.0
    p = np.bincount(y) / len(y)
    return 1.0 - np.sum(p ** 2)

def _entropy(y):
    if len(y) == 0: return 0.0
    p = np.bincount(y) / len(y)
    return -np.sum(p * np.log2(p + 1e-9))

# ── Node ──────────────────────────────────────────────────────────────────
class Node:
    """Internal or leaf node of a Decision Tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature   = feature
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value

    def is_leaf_node(self):
        return self.value is not None

# ── Decision Tree (base learner) ──────────────────────────────────────────
class DecisionTreeScratch:
    """
    Single Decision Tree used as a base learner inside Random Forest.
    Supports random feature subsampling via the `max_features` parameter.
    """
    def __init__(self, max_depth=10, min_samples_split=2,
                 criterion='gini', max_features=None, random_state=None):
        self.max_depth        = max_depth
        self.min_samples_split = min_samples_split
        self.criterion        = criterion
        self.max_features     = max_features   # None = use all features
        self.random_state     = random_state
        self.root             = None
        self._rng             = np.random.RandomState(random_state)

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        n_samples, n_feat = X.shape
        if depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1:
            return Node(value=int(np.bincount(y).argmax()))

        # Random feature subset
        m = self.max_features if self.max_features else n_feat
        feat_idxs = self._rng.choice(n_feat, m, replace=False)

        best_feat = best_thresh = None
        best_gain = -np.inf
        impurity_fn = _gini if self.criterion == 'gini' else _entropy
        parent_imp  = impurity_fn(y)

        for f in feat_idxs:
            for t in np.unique(X[:, f]):
                l = np.where(X[:, f] <= t)[0]
                r = np.where(X[:, f] >  t)[0]
                if len(l) == 0 or len(r) == 0: continue
                gain = parent_imp - (len(l)/n_samples)*impurity_fn(y[l]) - (len(r)/n_samples)*impurity_fn(y[r])
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, f, t

        if best_feat is None:
            return Node(value=int(np.bincount(y).argmax()))

        l_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        r_idx = np.where(X[:, best_feat] >  best_thresh)[0]
        return Node(feature=best_feat, threshold=best_thresh,
                    left =self._build_tree(X[l_idx], y[l_idx], depth+1),
                    right=self._build_tree(X[r_idx], y[r_idx], depth+1))

    def _traverse(self, x, node):
        if node.is_leaf_node(): return node.value
        if x[node.feature] <= node.threshold: return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

### 1.3. Bootstrap Sampling
Complete the function below.
Bootstrap sampling draws $N$ samples **with replacement** from the training set,
producing a new dataset of the same size each time.

In [ ]:
def bootstrap_sample(X, y, random_state=None):
    """
    Draw a bootstrap sample (with replacement) of size N from (X, y).

    Parameters
    ----------
    X            : np.ndarray, shape (N, p)
    y            : np.ndarray, shape (N,)
    random_state : int or None

    Returns
    -------
    X_sample, y_sample : np.ndarray
    """
    rng = np.random.RandomState(random_state)
    N   = X.shape[0]

    # TODO: Sample N indices with replacement from range [0, N)
    idxs = rng.choice(N, N, replace=True)  # Fill in your code here  Hint: rng.choice(N, N, replace=True)

    # TODO: Return the sampled rows from X and y
    return X[idxs], y[idxs]  # Fill in your code here


# --- Sanity check ---
np.random.seed(0)
_X = np.arange(10).reshape(5, 2)
_y = np.array([0, 1, 2, 0, 1])
_Xs, _ys = bootstrap_sample(_X, _y, random_state=42)
print('Original size :', _X.shape[0])
print('Bootstrap size:', _Xs.shape[0])
print('Sampled labels:', _ys)   # Some labels will repeat

### 1.4. Implementing RandomForestScratch
Complete the `# TODO` blocks inside `RandomForestScratch`.
The class must:
1. Train `n_estimators` Decision Trees, each on a different bootstrap sample.
2. At prediction time, aggregate tree votes via majority voting.

In [ ]:
class RandomForestScratch:
    """
    Random Forest classifier built from scratch.

    Parameters
    ----------
    n_estimators   : int   — number of Decision Trees in the ensemble
    max_depth      : int   — maximum depth of each tree
    min_samples_split : int — minimum samples to allow a node split
    max_features   : int or None — features considered per split (None = sqrt(p))
    criterion      : str   — 'gini' or 'entropy'
    random_state   : int or None
    """
    def __init__(self, n_estimators=100, max_depth=10, min_samples_split=2,
                 max_features=None, criterion='gini', random_state=None):
        self.n_estimators      = n_estimators
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.max_features      = max_features
        self.criterion         = criterion
        self.random_state      = random_state
        self.trees             = []   # stores trained DecisionTreeScratch objects

    # ── Training ─────────────────────────────────────────────────────────
    def fit(self, X, y):
        """Train n_estimators trees on bootstrap samples of (X, y)."""
        self.trees = []
        p = X.shape[1]

        # Default max_features = floor(sqrt(p)) for classification
        max_feat = self.max_features if self.max_features else int(np.floor(np.sqrt(p)))

        for i in range(self.n_estimators):
            # TODO: Draw a bootstrap sample from (X, y)
            #       Use seed = (self.random_state + i) if random_state is not None, else None
            seed     = (self.random_state + i) if self.random_state is not None else None
            X_bs, y_bs = bootstrap_sample(X, y, random_state=seed)  # Fill in your code here

            # TODO: Create and train a DecisionTreeScratch on the bootstrap sample
            #       Pass max_features=max_feat and random_state=seed
            tree = DecisionTreeScratch(max_depth=self.max_depth,
                                       min_samples_split=self.min_samples_split,
                                       criterion=self.criterion,
                                       max_features=max_feat,
                                       random_state=seed)  # Fill in your code here
            tree.fit(X_bs, y_bs)  # Fill in your code here  (tree.fit(...))

            # TODO: Append the trained tree to self.trees
            self.trees.append(tree)  # Fill in your code here

    # ── Prediction ───────────────────────────────────────────────────────
    def predict(self, X):
        """Aggregate predictions from all trees via majority vote."""
        # TODO: Collect predictions from each tree — shape (n_estimators, n_samples)
        all_preds = np.array([tree.predict(X) for tree in self.trees])  # Fill in your code here
        # Hint: np.array([tree.predict(X) for tree in self.trees])

        # TODO: For each sample, return the class with the most votes
        #       Hint: np.apply_along_axis + np.bincount.argmax, or scipy.stats.mode
        return np.apply_along_axis(lambda votes: np.bincount(votes).argmax(),
                                   axis=0, arr=all_preds)  # Fill in your code here

### 1.5. Loading the Iris Dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('Iris.csv').drop(columns=['Id'])
le = LabelEncoder()
df['Species'] = le.fit_transform(df['Species'])

X = df.drop(columns=['Species']).values
y = df['Species'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Classes          :', le.classes_)
print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')

### 1.6. Training RandomForestScratch and Comparing with Scikit-Learn
After completing all `# TODO` blocks above, run this cell to benchmark your
implementation against Scikit-Learn's `RandomForestClassifier`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- Custom Random Forest ---
rf_scratch = RandomForestScratch(n_estimators=50, max_depth=5,
                                  criterion='gini', random_state=42)
rf_scratch.fit(X_train, y_train)
y_pred_scratch = rf_scratch.predict(X_test)

# --- Scikit-Learn baseline ---
rf_sklearn = RandomForestClassifier(n_estimators=50, max_depth=5,
                                     criterion='gini', random_state=42)
rf_sklearn.fit(X_train, y_train)
y_pred_sklearn = rf_sklearn.predict(X_test)

acc_scratch = accuracy_score(y_test, y_pred_scratch)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

print('=== ACCURACY COMPARISON ON IRIS TEST SET ===')
print(f'RandomForestScratch         : {acc_scratch:.4f}')
print(f'RandomForestClassifier (sk) : {acc_sklearn:.4f}')

**Note**: A correct implementation should achieve accuracy close to Scikit-Learn.
Minor differences arise from Scikit-Learn's optimised split-search and tie-breaking rules.

### 1.7. Effect of Ensemble Size (n_estimators)
Observe how test accuracy changes as the number of trees in the forest grows.
This illustrates the key advantage of ensembling: variance reduction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

n_range = [1, 5, 10, 20, 30, 50, 75, 100]
accs_scratch = []
accs_sklearn = []

# Use 5-Fold CV on X_train to evaluate n_estimators and avoid test set peeking
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for n in n_range:
    scores_s = []
    scores_k = []
    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # Custom
        rf_s = RandomForestScratch(n_estimators=n, max_depth=5, random_state=42)
        rf_s.fit(X_tr, y_tr)
        scores_s.append(accuracy_score(y_val, rf_s.predict(X_val)))
        
        # Sklearn
        rf_k = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
        rf_k.fit(X_tr, y_tr)
        scores_k.append(accuracy_score(y_val, rf_k.predict(X_val)))
        
    accs_scratch.append(np.mean(scores_s))
    accs_sklearn.append(np.mean(scores_k))

plt.figure(figsize=(9, 4))
plt.plot(n_range, accs_scratch, marker='o', label='RandomForestScratch CV', linewidth=2)
plt.plot(n_range, accs_sklearn, marker='s', linestyle='--', label='RandomForestClassifier (sk) CV', linewidth=2)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Validation Accuracy (5-Fold CV)')
plt.title('CV Accuracy vs. Ensemble Size — Iris Dataset')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()


**Note**: Accuracy improves rapidly for small $T$ and stabilises as $T$ grows.
Beyond a threshold (typically $T \approx 50$–100 for Iris), additional trees
provide negligible accuracy gains while increasing training time.

### 1.8. Visualising the Decision Boundary
We project onto `PetalLengthCm` and `PetalWidthCm` to compare the
piecewise-rectangular boundaries of a single Decision Tree vs. the smoother
consensus boundary of the Random Forest.

In [ ]:
import matplotlib.colors as mcolors

FEAT_IDX  = [2, 3]   # PetalLengthCm, PetalWidthCm
FEAT_LABS = ['PetalLengthCm', 'PetalWidthCm']
COLORS    = ['#4e79a7', '#f28e2b', '#59a14f']
CMAP      = mcolors.ListedColormap(COLORS)

X2_tr = X_train[:, FEAT_IDX]
X2_te = X_test[:,  FEAT_IDX]

# Single Decision Tree baseline
from sklearn.tree import DecisionTreeClassifier
dt_single = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_single.fit(X2_tr, y_train)

# Random Forest (scratch, 2 features only)
rf_2d = RandomForestScratch(n_estimators=50, max_depth=5, max_features=2, random_state=42)
rf_2d.fit(X2_tr, y_train)

def plot_boundary(ax, model, X, y, title):
    h = 0.02
    x0_min, x0_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    x1_min, x1_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x0_min, x0_max, h),
                         np.arange(x1_min, x1_max, h))
    Z = np.array(model.predict(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
    for cls, col, lbl in zip([0, 1, 2], COLORS, le.classes_):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=col, edgecolors='k', s=40, label=lbl)
    ax.set_xlabel(FEAT_LABS[0])
    ax.set_ylabel(FEAT_LABS[1])
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.4)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_boundary(axes[0], dt_single, X2_te, y_test, 'Single Decision Tree (max_depth=5)')
try:
    plot_boundary(axes[1], rf_2d, X2_te, y_test, 'Random Forest — Scratch (50 trees)')
except Exception as e:
    axes[1].set_title('Complete implementation first')
    axes[1].text(0.5, 0.5, str(e), transform=axes[1].transAxes, ha='center', fontsize=8)
plt.suptitle('Iris Classification — Petal Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## PART 2: USING SCIKIT-LEARN ON THE IRIS DATASET

In this section we apply Scikit-Learn's `RandomForestClassifier` to `Iris.csv`,
performing full EDA, hyperparameter tuning, and ensemble-specific evaluation.

### 2.1. Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('Iris.csv')
df_raw.head(10)

In [ ]:
print('Column names:', df_raw.columns.tolist())
print(f'Shape: {df_raw.shape}')

In [ ]:
df_raw.describe().round(2)

**Note**: All features are continuous morphological measurements in centimetres.
Petal dimensions typically exhibit the highest inter-class variance and
are expected to dominate feature importance in tree-based models.

In [ ]:
print('Missing values:')
print(df_raw.isnull().sum())

**Note**: No missing values — no imputation required.

In [ ]:
print('Class distribution:')
print(df_raw['Species'].value_counts())
print()
print('Class proportions (%):')
print((df_raw['Species'].value_counts(normalize=True) * 100).round(2))

**Note**: The dataset is perfectly balanced (50 samples per class).
Accuracy is a reliable primary metric; no class-weighting adjustment is needed.

Visualise pairwise feature distributions coloured by species.

In [ ]:
sns.pairplot(df_raw.drop(columns=['Id']), hue='Species',
             palette={'Iris-setosa':'#4e79a7',
                      'Iris-versicolor':'#f28e2b',
                      'Iris-virginica':'#59a14f'},
             diag_kind='kde', plot_kws={'alpha': 0.7})
plt.suptitle('Pairwise Feature Distributions — Iris Dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Note**: `PetalLengthCm` and `PetalWidthCm` provide near-perfect linear separation for *Iris-setosa*
and strong class boundaries for the other two species.

### 2.2. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = df_raw.drop(columns=['Id'])
le = LabelEncoder()
y_enc = le.fit_transform(df['Species'])
X_all = df.drop(columns=['Species']).values
FEATURE_NAMES = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print('Classes          :', le.classes_)

### 2.3. Training Random Forest Classifier (Criterion Comparison)

#### 2.3.1. Gini Impurity Criterion

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_gini = RandomForestClassifier(n_estimators=100, criterion='gini',
                                   random_state=42, n_jobs=-1)
rf_gini.fit(X_train, y_train)
y_pred_gini = rf_gini.predict(X_test)

print(f'Accuracy (Gini): {accuracy_score(y_test, y_pred_gini):.4f}')
print(classification_report(y_test, y_pred_gini, target_names=le.classes_))

#### 2.3.2. Entropy Criterion

In [ ]:
rf_entropy = RandomForestClassifier(n_estimators=100, criterion='entropy',
                                      random_state=42, n_jobs=-1)
rf_entropy.fit(X_train, y_train)
y_pred_entropy = rf_entropy.predict(X_test)

print(f'Accuracy (Entropy): {accuracy_score(y_test, y_pred_entropy):.4f}')
print(classification_report(y_test, y_pred_entropy, target_names=le.classes_))

**Note**: Both criteria typically yield 95–100% accuracy on Iris.
Entropy can produce marginally different tree structures due to the logarithmic
sensitivity to small probability differences, but results are usually equivalent.

### 2.4. Cross-Validation and Hyperparameter Tuning

#### a) Tuning n_estimators via 5-Fold Cross-Validation
We scan `n_estimators` to observe when ensemble performance stabilises.

In [ ]:
from sklearn.model_selection import cross_val_score

n_range   = [1, 5, 10, 20, 50, 100, 150, 200]
cv_mean   = []
cv_std    = []

for n in n_range:
    rf     = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
    cv_mean.append(scores.mean())
    cv_std.append(scores.std())

best_n = n_range[np.argmax(cv_mean)]
print(f'Best n_estimators : {best_n}  |  CV Accuracy : {max(cv_mean):.4f}')

plt.figure(figsize=(9, 4))
plt.plot(n_range, cv_mean, marker='o', linewidth=2, label='Mean CV Accuracy')
plt.fill_between(n_range,
                 [m - s for m, s in zip(cv_mean, cv_std)],
                 [m + s for m, s in zip(cv_mean, cv_std)],
                 alpha=0.15, label='± 1 std')
plt.axvline(best_n, color='red', linestyle='--', label=f'Best = {best_n}')
plt.xlabel('n_estimators')
plt.ylabel('5-Fold CV Accuracy')
plt.title('Validation Curve: Accuracy vs n_estimators')
plt.legend()
plt.tight_layout()
plt.show()

**Note**: Accuracy stabilises rapidly on Iris.
The shaded band (± 1 std) narrows with larger $T$, confirming
that ensemble variance decreases as more trees are added.

#### b) Tuning max_features via 5-Fold Cross-Validation

In [ ]:
max_feat_options = ['sqrt', 'log2', 2, 3, 4]
cv_mf_mean = []

for mf in max_feat_options:
    rf     = RandomForestClassifier(n_estimators=100, max_features=mf,
                                     random_state=42, n_jobs=-1)
    scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
    cv_mf_mean.append(scores.mean())

x_labels = [str(mf) for mf in max_feat_options]
plt.figure(figsize=(7, 4))
plt.bar(x_labels, cv_mf_mean, color='#4e79a7', edgecolor='white')
plt.ylabel('5-Fold CV Accuracy')
plt.xlabel('max_features')
plt.title('Validation: Accuracy vs max_features')
plt.ylim(min(cv_mf_mean) - 0.01, 1.01)
plt.tight_layout()
plt.show()

for mf, acc in zip(x_labels, cv_mf_mean):
    print(f'  max_features={mf:5s}: CV Accuracy = {acc:.4f}')

**Note**: `max_features='sqrt'` (default) is generally the recommended starting point.
Larger feature subsets reduce randomness between trees, potentially increasing
inter-tree correlation and reducing the diversity benefit of bagging.

#### c) Tuning max_depth via 5-Fold Cross-Validation

In [ ]:
depth_options = [None, 2, 3, 5, 10]
cv_depth_mean = []

for d in depth_options:
    rf     = RandomForestClassifier(n_estimators=100, max_depth=d,
                                     random_state=42, n_jobs=-1)
    scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
    cv_depth_mean.append(scores.mean())

x_labels = [str(d) for d in depth_options]
plt.figure(figsize=(7, 4))
plt.bar(x_labels, cv_depth_mean, color='#f28e2b', edgecolor='white')
plt.ylabel('5-Fold CV Accuracy')
plt.xlabel('max_depth')
plt.title('Validation: Accuracy vs max_depth')
plt.ylim(min(cv_depth_mean) - 0.01, 1.01)
plt.tight_layout()
plt.show()

for d, acc in zip(x_labels, cv_depth_mean):
    print(f'  max_depth={d:4s}: CV Accuracy = {acc:.4f}')

**Note**: Unlike a single Decision Tree, a Random Forest is less sensitive to `max_depth`
because averaging over many trees already mitigates overfitting.
Fully grown trees (`max_depth=None`) often perform best.

### 2.5. Joint Hyperparameter Optimisation using RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'     : [50, 100, 150, 200],
    'max_depth'        : [None, 3, 5, 10],
    'max_features'     : ['sqrt', 'log2', 2, 3],
    'min_samples_split': [2, 5, 10],
    'criterion'        : ['gini', 'entropy'],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
random_search.fit(X_train, y_train)

print('Best parameters  :', random_search.best_params_)
print(f'Best CV Accuracy : {random_search.best_score_:.4f}')

### 2.6. Evaluating the Best Model on the Independent Test Set

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay
)

best_rf = random_search.best_estimator_
y_pred  = best_rf.predict(X_test)

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Best Random Forest')
plt.tight_layout()
plt.show()

#### OOB (Out-of-Bag) Score

In [ ]:
# OOB score uses the ~37% of samples NOT in each tree's bootstrap sample
# as a built-in validation set — no separate CV required
rf_oob = RandomForestClassifier(
    **random_search.best_params_,
    oob_score=True,
    random_state=42,
    n_jobs=-1,
)
rf_oob.fit(X_train, y_train)
print(f'OOB Score (training set internal estimate): {rf_oob.oob_score_:.4f}')
print(f'Test Accuracy                             : {accuracy_score(y_test, rf_oob.predict(X_test)):.4f}')

**Note**: The OOB score is an unbiased estimate of generalisation error computed without
a separate validation set, leveraging the ~37% of samples excluded from each bootstrap.

#### Feature Importances

In [ ]:
importances = best_rf.feature_importances_
feat_series = pd.Series(importances, index=FEATURE_NAMES).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
feat_series.plot(kind='barh', color='#59a14f', edgecolor='white')
plt.xlabel('Mean Decrease in Impurity (MDI)')
plt.title('Feature Importances — Best Random Forest')
plt.tight_layout()
plt.show()

print('Feature Importances:')
print(feat_series.sort_values(ascending=False).to_string())

**Note**: Random Forest feature importance is computed as the **Mean Decrease in Impurity (MDI)**
— the average impurity reduction weighted by the number of samples at each split across all trees.
`PetalLengthCm` and `PetalWidthCm` consistently dominate.

### 2.7. Comparing Random Forest vs Single Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_baseline = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_baseline.fit(X_train, y_train)
acc_dt = accuracy_score(y_test, dt_baseline.predict(X_test))
acc_rf = accuracy_score(y_test, best_rf.predict(X_test))

print('=== SINGLE TREE vs RANDOM FOREST ===')
print(f'Single Decision Tree (max_depth=5) : {acc_dt:.4f}')
print(f'Best Random Forest                  : {acc_rf:.4f}')

models = ['Decision Tree\n(max_depth=5)', 'Random Forest\n(Best Params)']
accs   = [acc_dt, acc_rf]
colors = ['#4e79a7', '#59a14f']

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accs, color=colors, edgecolor='white', width=0.4)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', fontsize=11, fontweight='bold')
plt.ylim(min(accs) - 0.05, 1.05)
plt.ylabel('Test Accuracy')
plt.title('Decision Tree vs Random Forest — Iris Dataset')
plt.tight_layout()
plt.show()

**Note**: The Random Forest consistently matches or outperforms the single Decision Tree
by aggregating diverse trees whose individual errors partially cancel out via majority voting.